In [1]:
!pip install openai-whisper pybind11


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.4 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=ce816826e110d9333fa55a875bb6d09e700ed191b8b19f7b9b5b2b92afabd778
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:
# Compile the C++ BPE tokenizer extension
!export EXT=$(python3 -c "import sysconfig; print(sysconfig.get_config_var('EXT_SUFFIX'))") && \
 export SRC="/kaggle/input/datasets/aneeshshastri/custom-tokenizers/bpe_tokenizer.cpp" && \
 g++ -O3 -Wall -shared -std=c++17 -fPIC $(python3 -m pybind11 --includes) $SRC -o /kaggle/working/bpe_tokenizer$EXT

!ls -l /kaggle/working/ | grep bpe_tokenizer


-rwxr-xr-x 1 root root 314544 May 11 09:13 bpe_tokenizer.cpython-312-x86_64-linux-gnu.so


In [3]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import pathlib
import librosa
import numpy as np
import math
import os
import whisper
import bpe_tokenizer
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42


In [4]:
# ── Constants & Configuration ─────────────────────────────────────────────────

NUM_CLASSES = 8
BATCH_SIZE  = 32

RAVDESS_INPUT      = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio"
TRANSCRIPTS_OUTPUT = "/kaggle/working/ravdess_transcripts"
CACHE_DIR          = "/kaggle/working/cache"
LOGITS_DIR         = "/kaggle/working/loso_logits"

EMOTION_MAP = {
    '01': 0,  # neutral
    '02': 1,  # calm
    '03': 2,  # happy
    '04': 3,  # sad
    '05': 4,  # angry
    '06': 5,  # fearful
    '07': 6,  # disgust
    '08': 7,  # surprised
}

MEL_CFG = dict(
    sr         = 22050,
    n_fft      = 1024,
    hop_length = 256,       # ~11.6 ms stride at 22 kHz
    n_mels     = 128,
    fmin       = 50,
    fmax       = 8000,
    duration   = 3.0,
)


In [5]:
# ── Data Parsing (with deduplication) ─────────────────────────────────────────

def parse_ravdess(root: str) -> list[dict]:
    """Parse RAVDESS filenames into records, skipping duplicate stems."""
    records = []
    seen_stems = set()

    for path in sorted(pathlib.Path(root).rglob("*.wav")):
        if path.stem in seen_stems:
            continue
        seen_stems.add(path.stem)

        parts = path.stem.split('-')
        records.append({
            "path":      str(path),
            "label":     EMOTION_MAP[parts[2]],
            "actor":     int(parts[6]),
            "intensity": int(parts[3]),
        })

    print(f"Parsed {len(records)} unique files "
          f"({len(seen_stems)} stems seen, duplicates skipped).")
    return records


In [6]:
# ── Whisper Transcription (with skip-if-done guard) ───────────────────────────

def transcribe_corpus(records: list[dict], output_path: str,
                      model_size: str = "base") -> str:
    """
    Transcribe audio files from parsed records using Whisper.
    Skips entirely if transcripts already exist on disk.
    """
    out_dir = Path(output_path)

    # Guard: reuse existing transcripts
    if out_dir.exists():
        existing = sorted(out_dir.glob("*.txt"))
        if len(existing) >= len(records):
            print(f"Found {len(existing)} existing transcripts in "
                  f"{output_path}. Skipping Whisper transcription.")
            return " ".join(f.read_text(encoding="utf-8") for f in existing)

    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Transcribing {len(records)} files with Whisper '{model_size}'...")
    model = whisper.load_model(model_size)
    corpus_texts = []

    for rec in tqdm(records, desc="Transcribing RAVDESS"):
        audio_path = Path(rec["path"])
        result = model.transcribe(
            str(audio_path),
            temperature=0.0,
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
        )
        transcription = result["text"].strip()
        (out_dir / f"{audio_path.stem}.txt").write_text(
            transcription, encoding="utf-8"
        )
        corpus_texts.append(transcription)

    return " ".join(corpus_texts)


In [7]:
# ── Audio Preprocessing ───────────────────────────────────────────────────────

def load_melspec(path: str, cfg: dict = MEL_CFG) -> np.ndarray:
    """Load a WAV file and return a log-mel spectrogram as (H, W, 1)."""
    wav, _ = librosa.load(path, sr=cfg["sr"], mono=True)
    target_len = int(cfg["sr"] * cfg["duration"])

    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    else:
        wav = wav[:target_len]

    mel = librosa.feature.melspectrogram(
        y=wav, sr=cfg["sr"], n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"], n_mels=cfg["n_mels"],
        fmin=cfg["fmin"], fmax=cfg["fmax"],
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)          # (n_mels, T)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    return log_mel[..., np.newaxis].astype(np.float32)       # (128, T, 1)


def preprocess_audio_cache(records: list[dict], cache_dir: str):
    """Compute mel spectrograms for all records, caching to disk."""
    cache = pathlib.Path(cache_dir)
    cache.mkdir(exist_ok=True)

    specs, labels = [], []
    for rec in tqdm(records, desc="Caching mel spectrograms"):
        npy_path = cache / (pathlib.Path(rec["path"]).stem + ".npy")
        if not npy_path.exists():
            mel = load_melspec(rec["path"])
            np.save(npy_path, mel)
        else:
            mel = np.load(npy_path)
        specs.append(mel)
        labels.append(rec["label"])

    X = np.stack(specs)
    y = np.array(labels, dtype=np.int32)
    np.save(cache / "X.npy", X)
    np.save(cache / "y.npy", y)
    return X, y


In [8]:
# ── Text Preprocessing ────────────────────────────────────────────────────────

def preprocess_transcriptions(tokenizer, root_dir: str):
    """
    Load transcript .txt files, BPE-encode them, and pad into
    a static-shaped matrix suitable for XLA / JAX.
    Returns (X_text, max_len, pad_id).
    """
    root_path = Path(root_dir)
    if not root_path.exists() or not root_path.is_dir():
        raise FileNotFoundError(f"Transcript directory not found: {root_dir}")

    txt_files = sorted(root_path.rglob("*.txt"))
    if not txt_files:
        raise ValueError(f"No .txt files found in {root_dir}")

    print(f"Discovered {len(txt_files)} transcript files. Encoding...")
    encoded_sequences = []
    for txt_path in tqdm(txt_files, desc="Encoding Text"):
        text = txt_path.read_text(encoding="utf-8").strip()
        encoded_sequences.append(tokenizer.encode(text))

    max_len = max(len(seq) for seq in encoded_sequences)
    pad_id  = tokenizer.get_vocab_size()   # pad token = vocab_size
    print(f"Max sequence length: {max_len} tokens | PAD_ID: {pad_id}")

    X_text = np.full((len(encoded_sequences), max_len), pad_id, dtype=np.int32)
    for i, seq in enumerate(encoded_sequences):
        X_text[i, :len(seq)] = seq

    return X_text, max_len, pad_id


In [9]:
# ── Execute Preprocessing Pipeline ────────────────────────────────────────────

# 1. Parse dataset (with dedup)
records = parse_ravdess(RAVDESS_INPUT)

# 2. Transcribe (skips if already done)
full_corpus_string = transcribe_corpus(records, TRANSCRIPTS_OUTPUT, model_size="base")
print(f"Total characters in corpus: {len(full_corpus_string)}")

# 3. Train BPE tokenizer on corpus
Tokenizer = bpe_tokenizer.BPETokenizer()
Tokenizer.train(full_corpus_string, num_merges=1_000)
VOCAB_SIZE = Tokenizer.get_vocab_size()
print(f"Vocab size: {VOCAB_SIZE}")

# 4. Cache mel spectrograms
X_audio, y = preprocess_audio_cache(records, CACHE_DIR)
print(f"X_audio shape: {X_audio.shape} | y shape: {y.shape}")

# 5. Encode transcripts
X_text, MAX_SEQ_LEN, PAD_ID = preprocess_transcriptions(Tokenizer, TRANSCRIPTS_OUTPUT)
print(f"X_text shape: {X_text.shape}")

# Sanity check: audio and text must have the same sample count
assert len(X_audio) == len(X_text) == len(y), (
    f"Mismatch: audio={len(X_audio)}, text={len(X_text)}, labels={len(y)}"
)


Parsed 1440 unique files (1440 stems seen, duplicates skipped).
Transcribing 1440 files with Whisper 'base'...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 121MiB/s]
Transcribing RAVDESS: 100%|██████████| 1440/1440 [06:48<00:00,  3.52it/s]


Total characters in corpus: 43347
Vocab size: 446


Caching mel spectrograms: 100%|██████████| 1440/1440 [00:31<00:00, 46.17it/s]


X_audio shape: (1440, 128, 259, 1) | y shape: (1440,)
Discovered 1440 transcript files. Encoding...


Encoding Text: 100%|██████████| 1440/1440 [00:00<00:00, 5547.02it/s]

Max sequence length: 12 tokens | PAD_ID: 446
X_text shape: (1440, 12)


In [10]:
# ── Train / Val / Test Split + VRAM Transfer ──────────────────────────────────

# First split: 80% train+val, 20% test
X_audio_tv, X_audio_test, X_text_tv, X_text_test, y_tv, y_test = train_test_split(
    X_audio, X_text, y, test_size=0.2, random_state=RANDOM_SEED,
)

# Second split: 90% train, 10% val (of the remaining 80%)
X_audio_train, X_audio_val, X_text_train, X_text_val, y_train, y_val = train_test_split(
    X_audio_tv, X_text_tv, y_tv, test_size=0.1, random_state=RANDOM_SEED,
)

# One-way transfer to GPU VRAM
X_audio_train = jax.device_put(X_audio_train)
X_audio_val   = jax.device_put(X_audio_val)
X_audio_test  = jax.device_put(X_audio_test)

X_text_train  = jax.device_put(X_text_train)
X_text_val    = jax.device_put(X_text_val)
X_text_test   = jax.device_put(X_text_test)

y_train = jax.device_put(y_train)
y_val   = jax.device_put(y_val)
y_test  = jax.device_put(y_test)

print(f"Train: audio {X_audio_train.shape}, text {X_text_train.shape}, labels {y_train.shape}")
print(f"Val:   audio {X_audio_val.shape},   text {X_text_val.shape},   labels {y_val.shape}")
print(f"Test:  audio {X_audio_test.shape},  text {X_text_test.shape},  labels {y_test.shape}")


Train: audio (1036, 128, 259, 1), text (1036, 12), labels (1036,)
Val:   audio (116, 128, 259, 1),   text (116, 12),   labels (116,)
Test:  audio (288, 128, 259, 1),  text (288, 12),  labels (288,)


In [11]:
# ── Batching Utility ──────────────────────────────────────────────────────────

def get_vram_batches(X, y, batch_size=BATCH_SIZE, shuffle=True, seed=42):
    """
    Yield {"input": ..., "labels": ...} batches from VRAM arrays.
    Drops the final incomplete batch for XLA-friendly static shapes.
    """
    num_samples = len(y)
    indices = np.arange(num_samples)

    if shuffle:
        np.random.default_rng(seed).shuffle(indices)

    num_batches = num_samples // batch_size
    for i in range(num_batches):
        idx = indices[i * batch_size : (i + 1) * batch_size]
        yield {"input": X[idx], "labels": y[idx]}


In [12]:
# ── Model Definitions ─────────────────────────────────────────────────────────


class OptimizedSpecAugment(nnx.Module):
    """Frequency + time masking for mel spectrograms (SpecAugment)."""

    def __init__(self, freq_mask_param: int, time_mask_param: int, rngs: nnx.Rngs):
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param
        # Store the full Rngs object so each forward pass draws a fresh key.
        # (Storing a single key would freeze the augmentation mask.)
        self.rngs = rngs

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        if not training:
            return x

        B, T_dim, F_dim = x.shape
        key = self.rngs.augmentation()
        key_f, key_f0, key_t, key_t0 = jax.random.split(key, 4)

        f  = jax.random.randint(key_f,  (B, 1, 1), 0, self.freq_mask_param)
        f0 = jax.random.randint(key_f0, (B, 1, 1), 0, F_dim)
        t  = jax.random.randint(key_t,  (B, 1, 1), 0, self.time_mask_param)
        t0 = jax.random.randint(key_t0, (B, 1, 1), 0, T_dim)

        freq_idx = jnp.arange(F_dim).reshape(1, 1, F_dim)
        time_idx = jnp.arange(T_dim).reshape(1, T_dim, 1)

        freq_mask = (freq_idx < f0) | (freq_idx >= f0 + f)
        time_mask = (time_idx < t0) | (time_idx >= t0 + t)

        x = jnp.where(freq_mask, x, 0.0)
        x = jnp.where(time_mask, x, 0.0)
        return x


# ── 1D Convolutional Block ───────────────────────────────────────────────────

class Conv1DBlock(nnx.Module):
    """1D Conv -> BatchNorm -> GELU -> MaxPool1D"""

    def __init__(self, in_features: int, out_features: int, rngs: nnx.Rngs):
        self.conv = nnx.Conv(
            in_features=in_features, out_features=out_features,
            kernel_size=(3,), padding="SAME", rngs=rngs,
        )
        self.bn = nnx.BatchNorm(num_features=out_features, rngs=rngs)

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        x = self.conv(x)
        x = self.bn(x, use_running_average=not training)
        x = jax.nn.gelu(x)
        x = jax.lax.reduce_window(
            x, -jnp.inf, jax.lax.max,
            window_dimensions=(1, 2, 1), window_strides=(1, 2, 1),
            padding="VALID",
        )
        return x


# ── 1D CNN Backbone ──────────────────────────────────────────────────────────

class CNN1DBackbone(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.block1 = Conv1DBlock(in_features=128, out_features=64,  rngs=rngs)
        self.block2 = Conv1DBlock(in_features=64,  out_features=128, rngs=rngs)
        self.block3 = Conv1DBlock(in_features=128, out_features=256, rngs=rngs)

    def __call__(self, x: jnp.ndarray, training: bool) -> jnp.ndarray:
        x = self.block1(x, training)   # (B, 129, 64)
        x = self.block2(x, training)   # (B, 64, 128)
        x = self.block3(x, training)   # (B, 32, 256)
        return x


# ── GRU Cell (kept for future experimentation) ──────────────────────────────
#
# class GRUCell(nnx.Module):
#     def __init__(self, input_size: int, hidden_size: int, rngs: nnx.Rngs):
#         self.Wr = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
#         self.Wz = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
#         self.Wh = nnx.Linear(input_size + hidden_size, hidden_size, rngs=rngs)
#         self.hidden_size = hidden_size
#
#     def __call__(self, x: jnp.ndarray, h: jnp.ndarray) -> jnp.ndarray:
#         xh = jnp.concatenate([x, h], axis=-1)
#         r  = jax.nn.sigmoid(self.Wr(xh))
#         z  = jax.nn.sigmoid(self.Wz(xh))
#         xh_reset = jnp.concatenate([x, r * h], axis=-1)
#         h_cand = jnp.tanh(self.Wh(xh_reset))
#         return (1 - z) * h + z * h_cand


# ── Audio Model: 1D-CRNN ────────────────────────────────────────────────────

class EmotionCRNN(nnx.Module):
    """1D CNN Backbone -> Global Max Pool -> Classifier"""

    def __init__(self, num_classes: int = NUM_CLASSES,
                 gru_hidden: int = 128, rngs: nnx.Rngs = None):
        self.spec_augment = OptimizedSpecAugment(
            freq_mask_param=20, time_mask_param=30, rngs=rngs,
        )
        self.cnn     = CNN1DBackbone(rngs)
        self.connect = nnx.Linear(256, gru_hidden, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.3, rngs=rngs)
        self.fc1     = nnx.Linear(gru_hidden, 64, rngs=rngs)
        self.fc2     = nnx.Linear(64, num_classes, rngs=rngs)
        self.gru_hidden = gru_hidden

    def __call__(self, x: jnp.ndarray, training: bool = True) -> jnp.ndarray:
        # Input: (B, 128, 259, 1)
        x = jnp.squeeze(x, axis=-1)        # (B, 128, 259)
        x = jnp.transpose(x, (0, 2, 1))    # (B, 259, 128)
        x = self.spec_augment(x, training=training)

        x = self.cnn(x, training)           # (B, 32, 256)

        # GRU path (kept commented for future experimentation)
        # B_dim = x.shape[0]
        # h = jnp.zeros((B_dim, self.gru_hidden))
        # def gru_step(h, x_t):
        #     h_new = self.gru(x_t, h)
        #     return h_new, h_new
        # h, _ = jax.lax.scan(gru_step, h, jnp.transpose(x, (1, 0, 2)))

        x = jnp.max(x, axis=1)             # Global max pool -> (B, 256)
        x = jax.nn.gelu(self.connect(x))
        x = self.dropout(x, deterministic=not training)
        x = jax.nn.gelu(self.fc1(x))
        return self.fc2(x)


# ── Text Model: GRU-based RNN ───────────────────────────────────────────────

class TextRNN(nnx.Module):
    def __init__(self, vocab_size: int, pad_id: int,
                 embed_dim: int, hidden_dim: int,
                 num_classes: int, rngs: nnx.Rngs):
        self.pad_id = pad_id
        self.embed  = nnx.Embed(num_embeddings=vocab_size + 1,
                                features=embed_dim, rngs=rngs)
        self.cell   = nnx.GRUCell(in_features=embed_dim,
                                  hidden_features=hidden_dim, rngs=rngs)
        self.rnn    = nnx.RNN(self.cell)
        self.dropout    = nnx.Dropout(rate=0.3, rngs=rngs)
        self.classifier = nnx.Linear(in_features=hidden_dim,
                                     out_features=num_classes, rngs=rngs)

    def __call__(self, x: jax.Array, training: bool = True) -> jax.Array:
        mask = (x != self.pad_id)
        emb  = self.embed(x)
        rnn_out = self.rnn(emb)

        # Masked mean pooling
        mask_exp    = jnp.expand_dims(mask, axis=-1)
        sum_hidden  = jnp.sum(rnn_out * mask_exp, axis=1)
        valid_lens  = jnp.maximum(jnp.sum(mask, axis=1, keepdims=True), 1)
        pooled      = sum_hidden / valid_lens

        x_drop = self.dropout(pooled, deterministic=not training)
        return self.classifier(x_drop)


In [13]:
# ── Training Utilities ────────────────────────────────────────────────────────
# ── Early Stopping with Best-Weight Restore ──────────────────────────────────

class EarlyStopping:
    """Monitors val loss, snapshots best weights, restores on stop."""
    def __init__(self, patience: int = 30, min_delta: float = 0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best_loss, self.wait = float("inf"), 0
        self.best_state, self.best_epoch, self.stopped_epoch = None, -1, -1

    def step(self, val_loss: float, model, epoch: int) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.wait, self.best_epoch = val_loss, 0, epoch
            _, self.best_state = nnx.split(model)
            return False
        self.wait += 1
        if self.wait >= self.patience:
            self.stopped_epoch = epoch
            return True
        return False

    def restore(self, model):
        if self.best_state is not None:
            nnx.update(model, self.best_state)
            print(f"Restored best weights from epoch {self.best_epoch} (val_loss={self.best_loss:.4f})")


# ── Class-Weighted Loss & Metrics ────────────────────────────────────────────

def compute_class_weights(labels: np.ndarray, num_classes: int = NUM_CLASSES) -> jnp.ndarray:
    """Inverse-frequency weights, normalised so they sum to num_classes."""
    counts = np.bincount(labels, minlength=num_classes).astype(np.float32)
    counts = np.maximum(counts, 1)          # avoid div-by-zero for absent classes
    inv_freq = 1.0 / counts
    weights = inv_freq / inv_freq.sum() * num_classes
    return jnp.array(weights)


def weighted_xent(logits, labels, class_weights):
    """Per-sample weighted softmax cross-entropy, averaged over the batch."""
    per_sample = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    sample_weights = class_weights[labels]
    return (per_sample * sample_weights).mean()


# ── Model + Optimizer Factories ──────────────────────────────────────────────

def create_audio_model_and_optimizer(lr: float = 3e-4):
    rngs  = nnx.Rngs(params=0, dropout=1, batch_stats=2, augmentation=3)
    model = EmotionCRNN(num_classes=NUM_CLASSES, gru_hidden=128, rngs=rngs)
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=lr,
        warmup_steps=200, decay_steps=5000, end_value=1e-6,
    )
    opt = nnx.Optimizer(model, optax.adamw(schedule, weight_decay=1e-4), wrt=nnx.Param)
    return model, opt


def create_text_model_and_optimizer(vocab_size: int, pad_id: int, lr: float = 3e-4):
    rngs  = nnx.Rngs(params=0, dropout=1)
    model = TextRNN(
        vocab_size=vocab_size, pad_id=pad_id,
        embed_dim=64, hidden_dim=128,
        num_classes=NUM_CLASSES, rngs=rngs,
    )
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=lr,
        warmup_steps=100, decay_steps=2000, end_value=1e-6,
    )
    opt = nnx.Optimizer(model, optax.adamw(schedule, weight_decay=1e-4), wrt=nnx.Param)
    return model, opt


# ── Unified step functions (weighted loss + weighted accuracy) ───────────────

@nnx.jit
def train_step(model, optimizer, metrics, batch, class_weights):
    def loss_fn(model):
        logits = model(batch["input"], training=True)
        loss = weighted_xent(logits, batch["labels"], class_weights)
        return loss, logits

    (loss, logits), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model, grads)

    preds   = jnp.argmax(logits, axis=-1)
    correct = (preds == batch["labels"]).astype(jnp.float32)
    sw      = class_weights[batch["labels"]]
    w_acc   = (correct * sw).sum() / sw.sum()

    metrics.update(loss=loss, weighted_acc=w_acc)
    return loss


@nnx.jit
def eval_step(model, metrics, batch, class_weights):
    logits = model(batch["input"], training=False)
    loss   = weighted_xent(logits, batch["labels"], class_weights)

    preds   = jnp.argmax(logits, axis=-1)
    correct = (preds == batch["labels"]).astype(jnp.float32)
    sw      = class_weights[batch["labels"]]
    w_acc   = (correct * sw).sum() / sw.sum()

    metrics.update(loss=loss, weighted_acc=w_acc)
    return loss


# ── Logit Collection ─────────────────────────────────────────────────────────

def collect_logits(model, X, batch_size=BATCH_SIZE):
    """Run inference on X and return raw logits as a NumPy array."""
    model.eval()
    parts = []
    n = len(X)
    for i in range(math.ceil(n / batch_size)):
        s, e = i * batch_size, min((i + 1) * batch_size, n)
        logits = model(X[s:e], training=False)
        parts.append(np.array(logits))
    return np.concatenate(parts, axis=0)


# ── Test-Set Evaluation (weighted) ───────────────────────────────────────────

def evaluate_test_set(model, X_test, y_test, class_weights, batch_size=BATCH_SIZE):
    """Evaluate on the held-out test set with weighted loss and balanced accuracy."""
    all_logits = collect_logits(model, X_test, batch_size)
    all_labels = np.array(y_test)

    # Weighted loss
    loss = float(weighted_xent(
        jnp.array(all_logits), jnp.array(all_labels), class_weights))

    # Balanced accuracy (mean per-class recall)
    preds = np.argmax(all_logits, axis=-1)
    recalls = []
    for c in range(NUM_CLASSES):
        mask = all_labels == c
        if mask.sum() > 0:
            recalls.append((preds[mask] == c).mean())
    balanced_acc = float(np.mean(recalls)) if recalls else 0.0
    regular_acc  = float((preds == all_labels).mean())

    print("-" * 40)
    print(f"  Weighted Loss:     {loss:.4f}")
    print(f"  Balanced Accuracy: {balanced_acc:.4f}")
    print(f"  Regular Accuracy:  {regular_acc:.4f}")
    print("-" * 40)
    return {"loss": loss, "balanced_acc": balanced_acc, "accuracy": regular_acc,
            "logits": all_logits}


In [14]:
# ── Audio Model Training (Weighted) ──────────────────────────────────────────

AUDIO_EPOCHS = 300
audio_model, audio_optimizer = create_audio_model_and_optimizer()

# Compute class weights from training labels
audio_cw = compute_class_weights(np.array(y_train))

metrics     = nnx.MultiMetric(loss=nnx.metrics.Average('loss'),
                               weighted_acc=nnx.metrics.Average('weighted_acc'))
val_metrics = nnx.MultiMetric(loss=nnx.metrics.Average('loss'),
                               weighted_acc=nnx.metrics.Average('weighted_acc'))
audio_es = EarlyStopping(patience=50)

print("Training EmotionCRNN (audio)...")
for epoch in range(AUDIO_EPOCHS):
    audio_model.train()
    for batch in get_vram_batches(X_audio_train, y_train, shuffle=True, seed=epoch):
        train_step(audio_model, audio_optimizer, metrics, batch, audio_cw)
    audio_model.eval()
    for batch in get_vram_batches(X_audio_val, y_val, shuffle=False):
        eval_step(audio_model, val_metrics, batch, audio_cw)
    vc = val_metrics.compute()
    print(f"Epoch {epoch:3d} | Train: {metrics.compute()} | Val: {vc}")
    metrics.reset(); val_metrics.reset()
    if audio_es.step(float(vc['loss']), audio_model, epoch):
        print(f"Early stopping at epoch {epoch}")
        break
audio_es.restore(audio_model)


Training EmotionCRNN (audio)...
Epoch   0 | Train: {'loss': Array(2.2113032, dtype=float32), 'weighted_acc': Array(0.13596213, dtype=float32)} | Val: {'loss': Array(2.0015593, dtype=float32), 'weighted_acc': Array(0.14064538, dtype=float32)}
Epoch   1 | Train: {'loss': Array(2.0959084, dtype=float32), 'weighted_acc': Array(0.12482953, dtype=float32)} | Val: {'loss': Array(2.0010977, dtype=float32), 'weighted_acc': Array(0.14064538, dtype=float32)}
Epoch   2 | Train: {'loss': Array(1.9842257, dtype=float32), 'weighted_acc': Array(0.14364281, dtype=float32)} | Val: {'loss': Array(2.000055, dtype=float32), 'weighted_acc': Array(0.14064538, dtype=float32)}
Epoch   3 | Train: {'loss': Array(1.9404708, dtype=float32), 'weighted_acc': Array(0.1639061, dtype=float32)} | Val: {'loss': Array(1.9942188, dtype=float32), 'weighted_acc': Array(0.14064538, dtype=float32)}
Epoch   4 | Train: {'loss': Array(1.8819317, dtype=float32), 'weighted_acc': Array(0.19222368, dtype=float32)} | Val: {'loss': Arr

In [15]:
# ── Text Model Training (Weighted) ───────────────────────────────────────────

TEXT_EPOCHS = 100
txt_model, txt_optimizer = create_text_model_and_optimizer(vocab_size=VOCAB_SIZE, pad_id=PAD_ID)
metrics.reset(); val_metrics.reset()
text_es = EarlyStopping(patience=20)

print("Training TextRNN...")
for epoch in range(TEXT_EPOCHS):
    txt_model.train()
    for batch in get_vram_batches(X_text_train, y_train, shuffle=True, seed=epoch):
        train_step(txt_model, txt_optimizer, metrics, batch, audio_cw)
    txt_model.eval()
    for batch in get_vram_batches(X_text_val, y_val, shuffle=False):
        eval_step(txt_model, val_metrics, batch, audio_cw)
    vc = val_metrics.compute()
    print(f"Epoch {epoch:3d} | Train: {metrics.compute()} | Val: {vc}")
    metrics.reset(); val_metrics.reset()
    if text_es.step(float(vc['loss']), txt_model, epoch):
        print(f"Early stopping at epoch {epoch}")
        break
text_es.restore(txt_model)


Training TextRNN...
Epoch   0 | Train: {'loss': Array(1.9530729, dtype=float32), 'weighted_acc': Array(0.13748242, dtype=float32)} | Val: {'loss': Array(2.0007539, dtype=float32), 'weighted_acc': Array(0.14635962, dtype=float32)}
Epoch   1 | Train: {'loss': Array(1.9565789, dtype=float32), 'weighted_acc': Array(0.13472266, dtype=float32)} | Val: {'loss': Array(2.0009432, dtype=float32), 'weighted_acc': Array(0.14259511, dtype=float32)}
Epoch   2 | Train: {'loss': Array(1.9500176, dtype=float32), 'weighted_acc': Array(0.13535026, dtype=float32)} | Val: {'loss': Array(2.0024316, dtype=float32), 'weighted_acc': Array(0.12360997, dtype=float32)}
Epoch   3 | Train: {'loss': Array(1.9517492, dtype=float32), 'weighted_acc': Array(0.11440743, dtype=float32)} | Val: {'loss': Array(2.0025134, dtype=float32), 'weighted_acc': Array(0.11338976, dtype=float32)}
Epoch   4 | Train: {'loss': Array(1.9525973, dtype=float32), 'weighted_acc': Array(0.13662915, dtype=float32)} | Val: {'loss': Array(2.00229

In [16]:
# ── Test Set Evaluation (Both Models, Weighted) ──────────────────────────────

test_cw = compute_class_weights(np.array(y_test))

print("=" * 40)
print("EmotionCRNN (Audio) — Test Results")
print("=" * 40)
evaluate_test_set(audio_model, X_audio_test, y_test, test_cw)

print()

print("=" * 40)
print("TextRNN — Test Results")
print("=" * 40)
evaluate_test_set(txt_model, X_text_test, y_test, test_cw)


EmotionCRNN (Audio) — Test Results
----------------------------------------
  Weighted Loss:     0.7973
  Balanced Accuracy: 0.7315
  Regular Accuracy:  0.7431
----------------------------------------

TextRNN — Test Results
----------------------------------------
  Weighted Loss:     1.9639
  Balanced Accuracy: 0.1001
  Regular Accuracy:  0.0799
----------------------------------------


{'loss': 1.9638515710830688,
 'balanced_acc': 0.10013899499193615,
 'accuracy': 0.0798611111111111,
 'logits': array([[-0.11883974,  0.04945926,  0.06554722, ..., -0.03965626,
          0.04383707,  0.05635416],
        [ 0.1129081 , -0.04270102, -0.01884871, ...,  0.04367105,
          0.02154004, -0.06714667],
        [-0.11883974,  0.04945926,  0.06554722, ..., -0.03965626,
          0.04383707,  0.05635416],
        ...,
        [-0.11883974,  0.04945926,  0.06554722, ..., -0.03965626,
          0.04383707,  0.05635416],
        [ 0.1129081 , -0.04270102, -0.01884871, ...,  0.04367105,
          0.02154004, -0.06714667],
        [-0.11883974,  0.04945926,  0.06554722, ..., -0.03965626,
          0.04383707,  0.05635416]], dtype=float32)}

In [17]:
# ── Leave-One-Subject-Out (LOSO) — Cascading Val + Logit Saving ──────────────

actors     = np.array([r["actor"] for r in records])
unique_ids = np.unique(actors)
print(f"LOSO evaluation over {len(unique_ids)} actors: {unique_ids.tolist()}\n")

os.makedirs(LOGITS_DIR, exist_ok=True)

# ── Cascading Validation Assignment (derangement) ────────────────────────────
# Each actor appears exactly once as a validation actor across all 24 folds.
rng = np.random.default_rng(RANDOM_SEED)
actors_list = unique_ids.tolist()
val_order = list(rng.permutation(actors_list))

# Fix self-assignments (val actor == test actor)
for i in range(len(actors_list)):
    if val_order[i] == actors_list[i]:
        j = (i + 1) % len(actors_list)
        val_order[i], val_order[j] = val_order[j], val_order[i]

val_assignments = dict(zip(actors_list, val_order))
print(f"Validation assignments: {val_assignments}\n")

per_actor_acc_audio = {}
per_actor_acc_text  = {}

for fold, held_out in enumerate(unique_ids):
    print(f"\n{'='*60}")
    print(f"  LOSO Fold {fold+1}/{len(unique_ids)} — held-out actor {held_out}")
    print(f"{'='*60}")

    val_actor = val_assignments[int(held_out)]
    print(f"  Validation actor: {val_actor}")

    test_mask  = (actors == held_out)
    val_mask   = (actors == val_actor)
    train_mask = ~(test_mask | val_mask)

    X_audio_fold_train = jax.device_put(X_audio[train_mask])
    X_audio_fold_val   = jax.device_put(X_audio[val_mask])
    X_audio_fold_test  = jax.device_put(X_audio[test_mask])
    X_text_fold_train  = jax.device_put(X_text[train_mask])
    X_text_fold_val    = jax.device_put(X_text[val_mask])
    X_text_fold_test   = jax.device_put(X_text[test_mask])
    y_fold_train       = jax.device_put(y[train_mask])
    y_fold_val         = jax.device_put(y[val_mask])
    y_fold_test        = jax.device_put(y[test_mask])

    print(f"  train {y_fold_train.shape[0]} | val {y_fold_val.shape[0]} | test {y_fold_test.shape[0]}")

    # Class weights from this fold's training labels
    fold_cw = compute_class_weights(np.array(y_fold_train))

    # ── Audio model ──────────────────────────────────────────────────────────
    fold_audio_model, fold_audio_opt = create_audio_model_and_optimizer()
    fm = nnx.MultiMetric(loss=nnx.metrics.Average('loss'),
                          weighted_acc=nnx.metrics.Average('weighted_acc'))
    fv = nnx.MultiMetric(loss=nnx.metrics.Average('loss'),
                          weighted_acc=nnx.metrics.Average('weighted_acc'))
    es_audio = EarlyStopping(patience=30)

    for epoch in range(AUDIO_EPOCHS):
        fold_audio_model.train()
        for batch in get_vram_batches(X_audio_fold_train, y_fold_train, shuffle=True, seed=epoch):
            train_step(fold_audio_model, fold_audio_opt, fm, batch, fold_cw)
        fold_audio_model.eval()
        for batch in get_vram_batches(X_audio_fold_val, y_fold_val, shuffle=False):
            eval_step(fold_audio_model, fv, batch, fold_cw)
        vc = fv.compute()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            print(f"  [Audio] Epoch {epoch+1:3d} | Train {fm.compute()} | Val {vc}")
        fm.reset(); fv.reset()
        if es_audio.step(float(vc['loss']), fold_audio_model, epoch):
            print(f"  [Audio] Early stop at epoch {epoch+1}")
            break
    es_audio.restore(fold_audio_model)

    audio_results = evaluate_test_set(fold_audio_model, X_audio_fold_test, y_fold_test, fold_cw)
    per_actor_acc_audio[int(held_out)] = audio_results["balanced_acc"]

    # Save audio logits (val + test)
    audio_val_logits  = collect_logits(fold_audio_model, X_audio_fold_val)
    audio_test_logits = audio_results["logits"]
    np.save(os.path.join(LOGITS_DIR, f"fold_{held_out}_audio_val_logits.npy"),  audio_val_logits)
    np.save(os.path.join(LOGITS_DIR, f"fold_{held_out}_audio_test_logits.npy"), audio_test_logits)

    # ── Text model ───────────────────────────────────────────────────────────
    fold_txt_model, fold_txt_opt = create_text_model_and_optimizer(vocab_size=VOCAB_SIZE, pad_id=PAD_ID)
    fm.reset(); fv.reset()
    es_text = EarlyStopping(patience=10)

    for epoch in range(TEXT_EPOCHS):
        fold_txt_model.train()
        for batch in get_vram_batches(X_text_fold_train, y_fold_train, shuffle=True, seed=epoch):
            train_step(fold_txt_model, fold_txt_opt, fm, batch, fold_cw)
        fold_txt_model.eval()
        for batch in get_vram_batches(X_text_fold_val, y_fold_val, shuffle=False):
            eval_step(fold_txt_model, fv, batch, fold_cw)
        vc = fv.compute()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            print(f"  [Text]  Epoch {epoch+1:3d} | Train {fm.compute()} | Val {vc}")
        fm.reset(); fv.reset()
        if es_text.step(float(vc['loss']), fold_txt_model, epoch):
            print(f"  [Text]  Early stop at epoch {epoch+1}")
            break
    es_text.restore(fold_txt_model)

    text_results = evaluate_test_set(fold_txt_model, X_text_fold_test, y_fold_test, fold_cw)
    per_actor_acc_text[int(held_out)] = text_results["balanced_acc"]

    # Save text logits (val + test)
    text_val_logits  = collect_logits(fold_txt_model, X_text_fold_val)
    text_test_logits = text_results["logits"]
    np.save(os.path.join(LOGITS_DIR, f"fold_{held_out}_text_val_logits.npy"),  text_val_logits)
    np.save(os.path.join(LOGITS_DIR, f"fold_{held_out}_text_test_logits.npy"), text_test_logits)

    # Save ground truth labels (val + test)
    np.save(os.path.join(LOGITS_DIR, f"fold_{held_out}_val_labels.npy"),  np.array(y_fold_val))
    np.save(os.path.join(LOGITS_DIR, f"fold_{held_out}_test_labels.npy"), np.array(y_fold_test))

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  LOSO Results — Per-Actor Balanced Accuracy")
print("=" * 60)
print(f"{'Actor':>7s}  {'Audio':>8s}  {'Text':>8s}")
print("-" * 30)
for aid in sorted(per_actor_acc_audio):
    print(f"{aid:>7d}  {per_actor_acc_audio[aid]:>8.4f}  {per_actor_acc_text[aid]:>8.4f}")
mean_a = np.mean(list(per_actor_acc_audio.values()))
mean_t = np.mean(list(per_actor_acc_text.values()))
print("-" * 30)
print(f"{'Mean':>7s}  {mean_a:>8.4f}  {mean_t:>8.4f}")
print("=" * 60)


LOSO evaluation over 24 actors: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]

Validation assignments: {1: np.int64(17), 2: np.int64(16), 3: np.int64(19), 4: np.int64(20), 5: np.int64(10), 6: np.int64(8), 7: np.int64(18), 8: np.int64(24), 9: np.int64(7), 10: np.int64(11), 11: np.int64(4), 12: np.int64(1), 13: np.int64(21), 14: np.int64(13), 15: np.int64(6), 16: np.int64(12), 17: np.int64(15), 18: np.int64(23), 19: np.int64(3), 20: np.int64(5), 21: np.int64(22), 22: np.int64(2), 23: np.int64(14), 24: np.int64(9)}


  LOSO Fold 1/24 — held-out actor 1
  Validation actor: 17
  train 1320 | val 60 | test 60
  [Audio] Epoch   1 | Train {'loss': Array(2.2000277, dtype=float32), 'weighted_acc': Array(0.13575639, dtype=float32)} | Val {'loss': Array(2.0817966, dtype=float32), 'weighted_acc': Array(0., dtype=float32)}
  [Audio] Epoch  25 | Train {'loss': Array(0.91139394, dtype=float32), 'weighted_acc': Array(0.6579818, dtype=float32)} | Val {'loss': Ar

In [18]:
# ── Late Fusion Model ────────────────────────────────────────────────────────

class LateFusion(nnx.Module):
    """Learns a scalar weight alpha to fuse softmax-normalised logits."""
    def __init__(self, rngs: nnx.Rngs):
        # sigmoid(0) = 0.5  →  equal initial weighting
        self.log_alpha = nnx.Param(jnp.array(0.0))

    def __call__(self, audio_probs: jnp.ndarray, text_probs: jnp.ndarray):
        alpha = jax.nn.sigmoid(self.log_alpha.value)
        return alpha * audio_probs + (1 - alpha) * text_probs


# ── Load & Concatenate All Validation Logits ─────────────────────────────────

all_audio_val, all_text_val, all_val_labels = [], [], []
for a in unique_ids:
    all_audio_val.append(np.load(os.path.join(LOGITS_DIR, f"fold_{a}_audio_val_logits.npy")))
    all_text_val.append(np.load(os.path.join(LOGITS_DIR, f"fold_{a}_text_val_logits.npy")))
    all_val_labels.append(np.load(os.path.join(LOGITS_DIR, f"fold_{a}_val_labels.npy")))

all_audio_val  = jnp.array(np.concatenate(all_audio_val))
all_text_val   = jnp.array(np.concatenate(all_text_val))
all_val_labels = jnp.array(np.concatenate(all_val_labels).astype(np.int32))

print(f"Fusion training data: {all_audio_val.shape[0]} samples")

# Softmax-normalise
all_audio_probs = jax.nn.softmax(all_audio_val, axis=-1)
all_text_probs  = jax.nn.softmax(all_text_val,  axis=-1)

# Class weights for the pooled validation set
fusion_cw = compute_class_weights(np.array(all_val_labels))

# ── Train the Fusion Model ───────────────────────────────────────────────────

fusion_model = LateFusion(rngs=nnx.Rngs(params=0))
fusion_opt   = nnx.Optimizer(fusion_model, optax.adam(1e-2), wrt=nnx.Param)

@nnx.jit
def fusion_train_step(model, opt, audio_p, text_p, labels, cw):
    def loss_fn(m):
        fused = m(audio_p, text_p)
        return weighted_xent(fused, labels, cw)
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    opt.update(model, grads)
    return loss

FUSION_EPOCHS = 500
for ep in range(FUSION_EPOCHS):
    loss = fusion_train_step(fusion_model, fusion_opt,
                             all_audio_probs, all_text_probs,
                             all_val_labels, fusion_cw)
    if (ep + 1) % 100 == 0 or ep == 0:
        alpha = float(jax.nn.sigmoid(fusion_model.log_alpha.value))
        print(f"  Fusion Epoch {ep+1:3d} | Loss {float(loss):.4f} | alpha={alpha:.4f}")

learned_alpha = float(jax.nn.sigmoid(fusion_model.log_alpha.value))
print(f"\nLearned fusion weight: alpha={learned_alpha:.4f}  (audio={learned_alpha:.2%}, text={1-learned_alpha:.2%})")


Fusion training data: 1440 samples
  Fusion Epoch   1 | Loss 1.7823 | alpha=0.5025
  Fusion Epoch 100 | Loss 1.7039 | alpha=0.7225
  Fusion Epoch 200 | Loss 1.6615 | alpha=0.8472
  Fusion Epoch 300 | Loss 1.6419 | alpha=0.9065
  Fusion Epoch 400 | Loss 1.6320 | alpha=0.9371
  Fusion Epoch 500 | Loss 1.6263 | alpha=0.9546

Learned fusion weight: alpha=0.9546  (audio=95.46%, text=4.54%)


In [19]:
# ── Fused Model LOSO Evaluation ──────────────────────────────────────────────
# Uses the saved test logits from Task 2 — no retraining needed.

per_actor_acc_fused = {}

print("\n" + "=" * 60)
print("  Fused Model LOSO Evaluation")
print("=" * 60)

for held_out in unique_ids:
    # Load saved test logits and labels
    audio_test = np.load(os.path.join(LOGITS_DIR, f"fold_{held_out}_audio_test_logits.npy"))
    text_test  = np.load(os.path.join(LOGITS_DIR, f"fold_{held_out}_text_test_logits.npy"))
    test_labels = np.load(os.path.join(LOGITS_DIR, f"fold_{held_out}_test_labels.npy"))

    # Softmax-normalise
    audio_probs = jax.nn.softmax(jnp.array(audio_test), axis=-1)
    text_probs  = jax.nn.softmax(jnp.array(text_test),  axis=-1)

    # Apply learned fusion
    fused_logits = np.array(fusion_model(audio_probs, text_probs))

    # Balanced accuracy
    preds = np.argmax(fused_logits, axis=-1)
    recalls = []
    for c in range(NUM_CLASSES):
        mask = test_labels == c
        if mask.sum() > 0:
            recalls.append((preds[mask] == c).mean())
    balanced_acc = float(np.mean(recalls)) if recalls else 0.0
    regular_acc  = float((preds == test_labels).mean())

    per_actor_acc_fused[int(held_out)] = balanced_acc
    print(f"  Actor {held_out:2d} | Balanced Acc: {balanced_acc:.4f} | Regular Acc: {regular_acc:.4f}")

# ── Final Summary ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  LOSO Results — All Models Comparison (Balanced Accuracy)")
print("=" * 60)
print(f"{'Actor':>7s}  {'Audio':>8s}  {'Text':>8s}  {'Fused':>8s}")
print("-" * 40)
for aid in sorted(per_actor_acc_audio):
    print(f"{aid:>7d}  {per_actor_acc_audio[aid]:>8.4f}  {per_actor_acc_text[aid]:>8.4f}  {per_actor_acc_fused[aid]:>8.4f}")
mean_a = np.mean(list(per_actor_acc_audio.values()))
mean_t = np.mean(list(per_actor_acc_text.values()))
mean_f = np.mean(list(per_actor_acc_fused.values()))
print("-" * 40)
print(f"{'Mean':>7s}  {mean_a:>8.4f}  {mean_t:>8.4f}  {mean_f:>8.4f}")
print("=" * 60)
print(f"\nLearned fusion alpha: {learned_alpha:.4f}")



  Fused Model LOSO Evaluation
  Actor  1 | Balanced Acc: 0.6719 | Regular Acc: 0.6500
  Actor  2 | Balanced Acc: 0.8281 | Regular Acc: 0.8167
  Actor  3 | Balanced Acc: 0.5625 | Regular Acc: 0.5667
  Actor  4 | Balanced Acc: 0.5469 | Regular Acc: 0.5500
  Actor  5 | Balanced Acc: 0.4844 | Regular Acc: 0.5167
  Actor  6 | Balanced Acc: 0.7031 | Regular Acc: 0.6833
  Actor  7 | Balanced Acc: 0.6719 | Regular Acc: 0.7167
  Actor  8 | Balanced Acc: 0.7656 | Regular Acc: 0.7833
  Actor  9 | Balanced Acc: 0.3281 | Regular Acc: 0.3500
  Actor 10 | Balanced Acc: 0.5625 | Regular Acc: 0.5667
  Actor 11 | Balanced Acc: 0.6406 | Regular Acc: 0.6500
  Actor 12 | Balanced Acc: 0.8438 | Regular Acc: 0.8333
  Actor 13 | Balanced Acc: 0.4062 | Regular Acc: 0.4000
  Actor 14 | Balanced Acc: 0.6875 | Regular Acc: 0.6667
  Actor 15 | Balanced Acc: 0.5781 | Regular Acc: 0.5833
  Actor 16 | Balanced Acc: 0.6250 | Regular Acc: 0.6500
  Actor 17 | Balanced Acc: 0.5312 | Regular Acc: 0.5667
  Actor 18 | Bala